In [1]:
!pip install PyGithub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.9 MB/s eta 0:00:00


In [2]:
from github import Github
from kaggle_secrets import UserSecretsClient
import os

In [3]:
import glob

GITHUB_REPO = "Zalanemoj/Stock-Market-Price-Prediction-Historical-Sentiments" 
REMOTE_BASE_FOLDER = "Data/" 

INPUT_BASE_DIR = '/kaggle/input/data-to-upload/'

FILES_TO_UPLOAD = glob.glob(os.path.join(INPUT_BASE_DIR, '**/*.csv'), recursive=True)

def upload_to_github(file_path, remote_file_path, repo_name):
    try:
        user_secrets = UserSecretsClient()
        access_token = user_secrets.get_secret("GITHUB_TOKEN")
    except:
        print("❌ Error: Could not find 'GITHUB_TOKEN' in Secrets.")
        return

    g = Github(access_token)
    repo = g.get_repo(repo_name)
    
    if not os.path.exists(file_path):
        print(f"❌ Error: File '{file_path}' does not exist.")
        return
        
    with open(file_path, 'rb') as file:
        content = file.read()
    try:
        contents = repo.get_contents(remote_file_path)
        repo.update_file(contents.path, f"Update {remote_file_path}", content, contents.sha)
        print(f"✅ Updated: {remote_file_path}")
    except:
        repo.create_file(remote_file_path, f"Add {remote_file_path}", content)
        print(f"✅ Created: {remote_file_path}")

if not FILES_TO_UPLOAD:
    print("⚠️ No CSV files found! Check your path.")
else:
    print(f"📂 Found {len(FILES_TO_UPLOAD)} files. Starting structured upload...")

    for local_file in FILES_TO_UPLOAD:
        relative_path = os.path.relpath(local_file, INPUT_BASE_DIR)
        destination_path = REMOTE_BASE_FOLDER + relative_path
        upload_to_github(local_file, destination_path, GITHUB_REPO)

⚠️ No CSV files found! Check your path.
